# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides an interactive walkthrough for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library, focusing on record sets, fields, and columns via their `@id`s.

### Dataset Source

The dataset Croissant schema is retrieved from:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and preview its description.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")


## 2. Data Overview

List all available record sets, their `@id`s, and constituent fields. It is important to reference all entities by their `@id`s as per Croissant best practices.

In [ ]:
# List all record sets and their fields by @id
print("Available Record Sets (by @id):")
record_sets = []
for rs in dataset.metadata.record_sets:
    print(f"  - {rs['@id']}: {rs['name'] if 'name' in rs else ''}")
    record_sets.append(rs['@id'])
    print("    Fields:")
    for field in rs['fields']:
        print(f"      * {field['@id']} : {field['name'] if 'name' in field else ''}")
    print()
if not record_sets:
    print("No explicit record sets found in top-level metadata. Trying infer from files...")
    # Try to infer if a 'main' record set exists (as sometimes used in Croissant when only one FileObject is present)
    if hasattr(dataset.metadata, 'record_sets') and dataset.metadata.record_sets:
        for rs in dataset.metadata.record_sets:
            print(f"  - {rs['@id']}")
    else:
        # Try using dataset.metadata.record_set if populated
        if hasattr(dataset.metadata, 'record_set') and dataset.metadata.record_set:
            if isinstance(dataset.metadata.record_set, list):
                for rs in dataset.metadata.record_set:
                    print(f"  - {rs['@id'] if isinstance(rs, dict) and '@id' in rs else str(rs)}")
            else:
                print(f"  - {dataset.metadata.record_set['@id'] if isinstance(dataset.metadata.record_set, dict) else str(dataset.metadata.record_set)}")
        else:
            print("No record set info available; refer to schema or mlcroissant docs for further inspection.")


## 3. Data Extraction

Load data from a specific record set (identified via its `@id`) into a pandas DataFrame. The first available record set and several of its fields will be demonstrated below. Adjust the `record_set_id` and field `@id`s as required for your workflow.

In [ ]:
# Identify ALL record set @ids:
record_set_ids = []
if hasattr(dataset.metadata, 'record_sets') and dataset.metadata.record_sets:
    for rs in dataset.metadata.record_sets:
        record_set_ids.append(rs['@id'])

# For this example, use the first available record set
if len(record_set_ids) > 0:
    record_set_id = record_set_ids[0]
    print(f"Using record set: {record_set_id}\n")
else:
    # Fallback: Use knowledge of possible Croissant patterns for a single table
    record_set_id = None

dataframes = {}

if record_set_id:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print("Fields in extracted DataFrame:")
    print(dataframes[record_set_id].columns.tolist())
    display(dataframes[record_set_id].head())
else:
    print("Record set not found or unavailable in metadata.")


## 4. Exploratory Data Analysis (EDA)

Select and process a numeric field (referenced by its `@id`). Apply standard filtering, normalization, and grouping.

If you do not know the available numeric field `@id`s, refer to the Data Overview section above to locate them.

In [ ]:
# Modify this section to point to a real numeric field and group (@id) from your dataset. Example values below:
# Suppose 'coverage_percent' and 'molecular_weight' are two numeric fields (replace with actual @id as necessary)

demo_numeric_field = None
demo_group_field = None

if record_set_id:
    # List field @id and names again for ease:
    columns = dataframes[record_set_id].columns
    print(f"Columns in DataFrame: {columns.tolist()}")
    # Try to pick a numeric one by heuristics
    for col in columns:
        if 'percent' in col or 'weight' in col or 'count' in col or 'abundance' in col or 'coverage' in col or dataframes[record_set_id][col].dtype.kind in 'if':
            demo_numeric_field = col
            break
    # Pick a group field (possible 'description', 'accession', or similar category)
    for col in columns:
        if col != demo_numeric_field and ('desc' in col or 'group' in col or 'type' in col or 'class' in col):
            demo_group_field = col
            break

    print(f"Chosen numeric field (by @id): {demo_numeric_field}")
    print(f"Chosen group field (by @id): {demo_group_field}")

    # Filter on numeric field
    if demo_numeric_field:
        # Clean non-numeric if present due to parsing
        df = dataframes[record_set_id]
        if not pd.api.types.is_numeric_dtype(df[demo_numeric_field]):
            df[demo_numeric_field] = pd.to_numeric(df[demo_numeric_field], errors='coerce')
        threshold = df[demo_numeric_field].mean() if pd.notnull(df[demo_numeric_field]).any() else 10
        filtered_df = df[df[demo_numeric_field] > threshold]
        print(f"Filtered records (where {demo_numeric_field} > {threshold:.2f}): {len(filtered_df)} records")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{demo_numeric_field}_normalized"] = (filtered_df[demo_numeric_field] - filtered_df[demo_numeric_field].mean()) / filtered_df[demo_numeric_field].std()
        print(f"Normalized {demo_numeric_field} for filtered records:")
        display(filtered_df[[demo_numeric_field, f"{demo_numeric_field}_normalized"]].head())

        # Group
        if demo_group_field and demo_group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(demo_group_field).mean(numeric_only=True)
            print(f"Mean statistics grouped by {demo_group_field}:")
            display(grouped_df.head())
else:
    print("No DataFrame available for EDA.")


## 5. Visualization

Visualize the distribution of the selected numeric field or the relationship between fields with matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and demo_numeric_field and demo_numeric_field in dataframes[record_set_id].columns:
    df = dataframes[record_set_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[demo_numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {demo_numeric_field}")
    plt.xlabel(demo_numeric_field)
    plt.ylabel('Count')
    plt.show()

    if demo_group_field and demo_group_field in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=demo_group_field, y=demo_numeric_field, data=df)
        plt.xticks(rotation=45)
        plt.title(f"{demo_numeric_field} by {demo_group_field}")
        plt.show()
else:
    print("Not enough data to plot. Ensure at least a record set and numeric field were found.")


## 6. Conclusion

- We loaded and inspected the FAIR² protein dataset as a Croissant package, dynamically referencing all entities and fields by `@id`.
- Key record sets and numeric fields were programmatically identified for initial EDA, including filtering and normalization of data.
- Visualization highlighted the distribution and grouped statistics of one numeric feature.

Next steps: Examine specific record sets for more advanced downstream tasks (e.g., building machine learning models, detailed analysis, etc.). See the [mlcroissant documentation](https://mlcommons.github.io/croissant/) for additional options.